# 三味線譜変換ツール

五線譜（PDF）→ MusicXML（oemer）→ 三味線中間XML の変換を行います。

**実行順序:** セル0（セットアップ）→ セル1（PDF変換）→ セル2（調弦選択）→ セル3（変換）→ セル4（ダウンロード）

In [ ]:
# ============================================================
# セル0: セットアップ（最初に一度だけ実行 / 再実行で最新版に更新）
# ============================================================

import subprocess, sys, os
from google.colab import userdata

REPO_NAME = "kamex120/shamisen-converter"
REPO_DIR  = "shamisen-converter"

def run(cmd, **kwargs):
    r = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if r.stdout: print(r.stdout.strip())
    if r.returncode != 0:
        print("❌ エラー:", r.stderr.strip())
        raise RuntimeError(f"コマンド失敗: {' '.join(cmd)}")
    return r

# リポジトリのクローン or 更新
token = userdata.get("GITHUB_TOKEN")
repo_url = f"https://{token}@github.com/{REPO_NAME}.git"
if not os.path.exists(REPO_DIR):
    print("📥 リポジトリをクローン中...")
    run(["git", "clone", repo_url, REPO_DIR])
else:
    print("🔄 最新版に更新中...")
    run(["git", "pull"], cwd=REPO_DIR)

os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f"📁 作業ディレクトリ: {os.getcwd()}")

# 依存ライブラリのインストール
print("\n📦 ライブラリをインストール中...")
run([sys.executable, "-m", "pip", "install", "music21", "pyyaml", "-q"])

print("\n✅ セットアップ完了")

In [ ]:
# ============================================================
# セル0-b: Audiveris セットアップ（Audiverisを使う場合のみ実行）
# ============================================================

import subprocess, os

DEB_URL = "https://github.com/Audiveris/audiveris/releases/download/5.10.2/Audiveris-5.10.2-ubuntu22.04-x86_64.deb"
DEB_PATH = "/tmp/audiveris.deb"
KNOWN_BIN = "/opt/audiveris/bin/Audiveris"  # .deb インストール時の既知パス

def _find_audiveris():
    """インストール済みの Audiveris バイナリパスを返す。見つからなければ None。"""
    r = subprocess.run(["which", "Audiveris"], capture_output=True, text=True)
    if r.returncode == 0 and r.stdout.strip():
        return r.stdout.strip()
    if os.path.isfile(KNOWN_BIN):
        return KNOWN_BIN
    return None

AUDIVERIS_BIN = _find_audiveris()

if AUDIVERIS_BIN:
    print(f"✅ Audiveris インストール済み: {AUDIVERIS_BIN}")
else:
    print("🔧 xvfb をインストール中...")
    subprocess.run(["apt-get", "install", "-y", "xvfb", "-q"], capture_output=True)

    print("📥 Audiveris をダウンロード中...")
    subprocess.run(["wget", "-q", "--show-progress", DEB_URL, "-O", DEB_PATH], check=True)

    print("📦 インストール中...")
    subprocess.run(["dpkg", "-i", DEB_PATH])
    subprocess.run(["apt-get", "install", "-f", "-y", "-q"], capture_output=True)
    os.remove(DEB_PATH)

    AUDIVERIS_BIN = _find_audiveris()
    if AUDIVERIS_BIN:
        print(f"✅ Audiveris セットアップ完了: {AUDIVERIS_BIN}")
    else:
        print("❌ Audiveris バイナリが見つかりません。インストールログを確認してください。")

---
## Step 1: PDF → MusicXML

In [ ]:
# ============================================================
# セル1-a: PDFをアップロード
# ============================================================

from google.colab import files

print("PDFファイルを選択してください")
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print(f"\n📄 アップロード: {pdf_path}")

In [ ]:
# ============================================================
# セル1-b: PDF → MusicXML（oemer または Audiveris）
# ============================================================

import os, subprocess, glob
from pdf2image import convert_from_path
from music21 import converter, stream
from tqdm.notebook import tqdm

OMR_DIR = "_omr_output"
os.makedirs(OMR_DIR, exist_ok=True)
omr_tool = omr_widget.value
print(f"OMRツール: {omr_tool}")

# -------------------------------------------------------
# oemer
# -------------------------------------------------------
if omr_tool == "oemer":
    IMG_DIR = "_pages"
    os.makedirs(IMG_DIR, exist_ok=True)

    print("📃 PDFを画像に変換中...")
    pages = convert_from_path(pdf_path, dpi=300)
    img_paths = []
    for i, page in enumerate(tqdm(pages, desc="PDF→画像")):
        img_path = f"{IMG_DIR}/page_{i+1:02d}.png"
        page.save(img_path, "PNG")
        img_paths.append(img_path)
    print(f"  → {len(img_paths)} ページ")

    print("\n🎵 oemer で楽譜認識中（ONNX Runtime + GPU）...")
    xml_paths = []
    for img_path in tqdm(img_paths, desc="楽譜認識"):
        basename = os.path.splitext(os.path.basename(img_path))[0]
        out_dir = f"{OMR_DIR}/{basename}"
        os.makedirs(out_dir, exist_ok=True)
        tqdm.write(f"  処理中: {img_path}")
        proc = subprocess.run(
            ["oemer", img_path, "-o", out_dir],
            capture_output=True, text=True
        )
        if proc.returncode != 0:
            tqdm.write(f"  ❌ エラー:\n{proc.stderr[-300:]}")
        found = [
            os.path.join(out_dir, f) for f in os.listdir(out_dir)
            if f.endswith(".xml") or f.endswith(".musicxml")
        ]
        if found:
            xml_paths.append(found[0])
            tqdm.write(f"  ✅ → {found[0]}")
        else:
            tqdm.write(f"  ⚠️  出力なし: {os.listdir(out_dir)}")

# -------------------------------------------------------
# Audiveris
# -------------------------------------------------------
else:
    if not AUDIVERIS_BIN:
        raise RuntimeError("Audiveris が見つかりません。セル0-b を先に実行してください。")

    out_dir = f"{OMR_DIR}/audiveris"
    os.makedirs(out_dir, exist_ok=True)
    print(f"🎵 Audiveris で楽譜認識中... ({AUDIVERIS_BIN})")

    proc = subprocess.run(
        ["xvfb-run", AUDIVERIS_BIN,
         "-batch", "-export", "-output", out_dir, "--", pdf_path],
        capture_output=True, text=True
    )
    if proc.returncode != 0:
        print("❌ Audiveris エラー:", proc.stderr[-500:])
    else:
        print("stdout:", proc.stdout[-300:])

    found_mxl = glob.glob(f"{out_dir}/**/*.mxl", recursive=True)
    found_xml = glob.glob(f"{out_dir}/**/*.xml", recursive=True)
    xml_paths = found_mxl + found_xml
    if xml_paths:
        print(f"  ✅ → {xml_paths[0]}")
    else:
        print(f"  ⚠️  出力なし: {os.listdir(out_dir)}")

# -------------------------------------------------------
# 共通：複数ページのマージ
# -------------------------------------------------------
if not xml_paths:
    raise RuntimeError("MusicXMLが1つも生成されませんでした。上の出力を確認してください。")

if len(xml_paths) == 1:
    musicxml_path = xml_paths[0]
else:
    print("\n📎 複数ページをマージ中...")
    merged = stream.Score()
    for xp in tqdm(xml_paths, desc="マージ"):
        sc = converter.parse(xp)
        for part in sc.parts:
            merged.append(part)
    musicxml_path = f"{OMR_DIR}/merged.xml"
    merged.write("musicxml", fp=musicxml_path)

print(f"\n✅ MusicXML生成完了 → {musicxml_path}")

In [ ]:
# ============================================================
# セル1-b: PDF → MusicXML（Audiveris）
# ============================================================

import os, subprocess, glob
from music21 import converter, stream

if not AUDIVERIS_BIN:
    raise RuntimeError("Audiveris が見つかりません。セル0-b を先に実行してください。")

OMR_DIR = "_omr_output/audiveris"
os.makedirs(OMR_DIR, exist_ok=True)
print(f"🎵 Audiveris で楽譜認識中...")

proc = subprocess.run(
    ["xvfb-run", AUDIVERIS_BIN,
     "-batch", "-export", "-output", OMR_DIR, "--", pdf_path],
    capture_output=True, text=True
)
if proc.returncode != 0:
    print("❌ Audiveris エラー:", proc.stderr[-500:])
else:
    print(proc.stdout[-300:] or "（出力なし）")

found_mxl = glob.glob(f"{OMR_DIR}/**/*.mxl", recursive=True)
found_xml  = glob.glob(f"{OMR_DIR}/**/*.xml",  recursive=True)
xml_paths  = found_mxl + found_xml

if not xml_paths:
    raise RuntimeError(f"MusicXML が生成されませんでした。出力フォルダ: {os.listdir(OMR_DIR)}")

if len(xml_paths) == 1:
    musicxml_path = xml_paths[0]
else:
    print(f"\n📎 {len(xml_paths)} ページをマージ中...")
    merged = stream.Score()
    for xp in xml_paths:
        for part in converter.parse(xp).parts:
            merged.append(part)
    musicxml_path = "_omr_output/merged.xml"
    merged.write("musicxml", fp=musicxml_path)

print(f"\n✅ MusicXML生成完了 → {musicxml_path}")

---
## Step 2: 調弦を選択

In [ ]:
# ============================================================
# セル2: 調弦選択
# ============================================================

import ipywidgets as widgets
from IPython.display import display

tuning_widget = widgets.ToggleButtons(
    options=[
        ("本調子", "honchoshi"),
        ("二上り", "niagari"),
        ("三下り", "sansagari"),
    ],
    description="調弦:",
    button_style="info",
)
display(tuning_widget)

---
## Step 3: MusicXML → 三味線中間XML

In [ ]:
# ============================================================
# セル3-a: 変換実行
# ============================================================

from shamisen_converter import (
    convert_musicxml, resolve_out_of_range,
    load_mapping, build_midi_to_position, to_intermediate_xml
)

tuning = tuning_widget.value
print(f"調弦: {tuning_widget.label} ({tuning})")

result = convert_musicxml(
    musicxml_path=musicxml_path,
    mapping_path="shamisen_mapping.yaml",
    tuning=tuning,
)

total = len([n for n in result.notes if n.note_name != "rest"])
out_of_range = len([n for n in result.notes if n.out_of_range])
print(f"\n音符数: {total}")
print(f"音域外: {out_of_range} 件")
if result.warnings:
    print(f"警告数: {len(result.warnings)} 件")

In [ ]:
# ============================================================
# セル3-a: 変換実行
# ============================================================

from shamisen_converter import (
    convert_musicxml, resolve_out_of_range,
    load_mapping, build_midi_to_position,
)

tuning = tuning_widget.value
print(f"調弦: {tuning_widget.label} ({tuning})")

result = convert_musicxml(
    musicxml_path=musicxml_path,
    mapping_path="shamisen_mapping.yaml",
    tuning=tuning,
)

total = len([n for n in result.notes if n.note_name != "rest"])
out_of_range = len([n for n in result.notes if n.out_of_range])
print(f"\n音符数: {total}")
print(f"音域外: {out_of_range} 件")
if result.warnings:
    print(f"警告数: {len(result.warnings)} 件")

In [ ]:
# ============================================================
# セル3-c: 中間XML出力
# ============================================================

OUTPUT_PATH = "shamisen_output.xml"

xml_str = to_intermediate_xml(result)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write(xml_str)

print(f"✅ 変換完了 → {OUTPUT_PATH}")

# 先頭30行をプレビュー
lines = xml_str.splitlines()
print("\n--- プレビュー（先頭30行）---")
print("\n".join(lines[:30]))

# ============================================================
# セル3-c: 中間YAML出力
# ============================================================

from shamisen_converter import to_intermediate_yaml

OUTPUT_PATH = "shamisen_output.yaml"

yaml_str = to_intermediate_yaml(result)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write(yaml_str)

print(f"✅ 変換完了 → {OUTPUT_PATH}")

In [ ]:
# ============================================================
# セル4: ダウンロード
# ============================================================

from google.colab import files
files.download(OUTPUT_PATH)
print(f"⬇️  {OUTPUT_PATH} をダウンロードしました")

# ============================================================
# セル4: ダウンロード
# ============================================================

from google.colab import files
files.download(OUTPUT_PATH)
print(f"⬇️  {OUTPUT_PATH} をダウンロードしました")

In [ ]:
# ============================================================
# オプション: MusicXMLを直接アップロード
# ============================================================

from google.colab import files

print("MusicXMLファイル（.xml / .mxl）を選択してください")
uploaded_xml = files.upload()
musicxml_path = list(uploaded_xml.keys())[0]
print(f"\n📄 アップロード: {musicxml_path}")
print("→ セル2（調弦選択）から続けてください")